In [ ]:
import rasterio
from rasterio.windows import from_bounds
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. CONFIGURATION & FILE PATHS
# ==========================================
# List all your Sentinel-2 tiles (Features)
sentinel_files = [
    "Sentinel2_Karnataka_2021-0000000000-0000011776.tif",
    "Sentinel2_Karnataka_2021-0000011776-0000000000.tif",
    "Sentinel2_Karnataka_2021-0000011776-0000011776.tif"
]

# The Reference Map (Labels)
label_file = "WorldCover_Karnataka_2021.tif"

# Define Class Names for better reports (Standard ESA WorldCover classes)
class_names = {
    10: "Tree Cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
    50: "Built-up", 60: "Bare / sparse veg", 70: "Snow and Ice",
    80: "Permanent water bodies", 90: "Herbaceous wetland", 95: "Mangroves", 100: "Moss and lichen"
}

# ==========================================
# 2. DATA LOADING & SPATIAL ALIGNMENT
# ==========================================
X_list = []
y_list = []

print("🚀 Starting Data Processing...")

with rasterio.open(label_file) as src_label:
    for s_file in sentinel_files:
        try:
            with rasterio.open(s_file) as src_img:
                print(f"   Processing tile: {s_file}")

                # A. Read Sentinel Data (Features)
                # Shape: (Bands, Height, Width) -> Transpose to (Pixels, Bands)
                img_data = src_img.read()
                bands, height, width = img_data.shape

                # Reshape to 2D array: (Number of Pixels, Number of Bands)
                X_tile = img_data.reshape(bands, -1).T

                # B. Read Corresponding Labels (Ground Truth)
                # We use the bounds of the Sentinel image to grab the EXACT matching window from the huge Label map
                bounds = src_img.bounds
                window = from_bounds(bounds.left, bounds.bottom, bounds.right, bounds.top, src_label.transform)

                # Read the label data for that specific window
                y_tile = src_label.read(1, window=window, out_shape=(height, width), boundless=True, fill_value=0)
                y_tile_flat = y_tile.flatten()

                # C. Data Cleaning (The "Mask")
                # We want to keep pixels where:
                # 1. Label is not 0 (0 usually means No Data in WorldCover)
                # 2. Sentinel Image is not 0 (0 usually means black border/nodata)
                valid_mask = (y_tile_flat > 0) & (np.max(X_tile, axis=1) > 0)

                # D. Append valid data to our lists
                X_list.append(X_tile[valid_mask])
                y_list.append(y_tile_flat[valid_mask])

        except Exception as e:
            print(f"⚠️ Error processing {s_file}: {e}")

# ==========================================
# 3. MERGE & PREPROCESS
# ==========================================
print("\n🔄 Merging data from all tiles...")
X_data = np.vstack(X_list)
y_data = np.concatenate(y_list)

print(f"   Total valid pixels found: {X_data.shape[0]:,}")

# OPTIONAL: Downsample if dataset is too huge (e.g., > 1 million pixels)
# This prevents your computer from freezing during training.
MAX_SAMPLES = 500_000
if X_data.shape[0] > MAX_SAMPLES:
    print(f"   ⚠️ Dataset too large. Downsampling to {MAX_SAMPLES:,} random pixels for training...")
    idx = np.random.choice(len(y_data), MAX_SAMPLES, replace=False)
    X_data = X_data[idx]
    y_data = y_data[idx]

# ==========================================
# 4. TRAIN / TEST SPLIT
# ==========================================
print("\n✂️ Splitting data into Training (80%) and Testing (20%)...")
# Stratify ensures we have a fair mix of all classes (water, forest, urban) in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.20, random_state=42, stratify=y_data
)

# ==========================================
# 5. MODEL TRAINING
# ==========================================
print(f"\n🧠 Training Random Forest Classifier on {len(X_train):,} pixels...")
rf_model = RandomForestClassifier(
    n_estimators=100,  # Number of trees
    n_jobs=-1,         # Use all CPU cores
    random_state=42,
    verbose=1
)
rf_model.fit(X_train, y_train)
print("   Training Complete.")

# ==========================================
# 6. VALIDATION & REPORTING
# ==========================================
print("\nabc Predicting on Test Set...")
y_pred = rf_model.predict(X_test)

# A. Overall Accuracy
acc = accuracy_score(y_test, y_pred)
print("-" * 60)
print(f"✅ FINAL ACCURACY: {acc * 100:.2f}%")
print("-" * 60)

# B. Detailed Classification Report
# Get unique classes present in the test set to map names correctly
unique_labels = np.unique(y_test)
target_names = [class_names.get(lbl, f"Class {lbl}") for lbl in unique_labels]

print("\n📊 Detailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

# C. Confusion Matrix (Optional visualization)
print("\nvm Confusion Matrix (Rows=True, Cols=Predicted):")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Hint for the user
print("\n💡 NOTE: If accuracy is between 85-98%, your model is Excellent.")
print("   If accuracy is 100.0%, check for data leakage (overlapping train/test areas).")

In [ ]:
import rasterio
from rasterio.windows import from_bounds
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

# ==========================================
# 1. FILE SETUP (Including the missing files)
# ==========================================
# I have added the 4th tile name based on the pattern of your other files.
# Make sure this file actually exists in your folder!
sentinel_files = [
    "Sentinel2_Karnataka_2021-0000000000-0000011776.tif",
    "Sentinel2_Karnataka_2021-0000011776-0000000000.tif",
    "Sentinel2_Karnataka_2021-0000011776-0000011776.tif",
    "Sentinel2_Karnataka_2021-0000000000-0000000000.tif" # <--- LIKELY YOUR MISSING 4TH TILE
]

# We will try the Karnataka label file first. If a tile falls outside it,
# we fallback to the larger SouthIndia file.
label_files = [
    "WorldCover_Karnataka_2021.tif",
    "Sentinel2_Karnataka_2021.tif"
]

# ==========================================
# 2. DATA PROCESSING LOOP
# ==========================================
X_list = []
y_list = []

print("🚀 Starting Data Extraction...")

for s_file in sentinel_files:
    try:
        with rasterio.open(s_file) as src_img:
            print(f"\nProcessing Feature Tile: {s_file}")

            # Read Sentinel Data
            img_data = src_img.read()
            bands, height, width = img_data.shape
            X_tile = img_data.reshape(bands, -1).T
            bounds = src_img.bounds

            # Try to find matching labels in our label files
            found_match = False
            for l_file in label_files:
                with rasterio.open(l_file) as src_label:
                    # Check if this sentinel tile overlaps with this label file
                    try:
                        # Grab the window of the label file that matches the satellite image
                        window = from_bounds(bounds.left, bounds.bottom, bounds.right, bounds.top, src_label.transform)

                        # Read labels
                        y_tile = src_label.read(1, window=window, out_shape=(height, width), boundless=True, fill_value=0)

                        # Check if we actually got valid data (not just empty zeros)
                        if np.any(y_tile > 0):
                            y_tile_flat = y_tile.flatten()

                            # Valid Mask: Remove Black Borders (0) and No-Data Labels (0)
                            valid_mask = (y_tile_flat > 0) & (np.max(X_tile, axis=1) > 0)

                            X_list.append(X_tile[valid_mask])
                            y_list.append(y_tile_flat[valid_mask])

                            print(f"   ✅ matched with {l_file} | Pixels: {np.sum(valid_mask)}")
                            found_match = True
                            break # Stop looking for labels, we found them
                    except Exception:
                        continue

            if not found_match:
                print("   ⚠️ Could not find matching label data for this tile.")

    except FileNotFoundError:
        print(f"   ❌ File not found: {s_file} (Skipping...)")

# ==========================================
# 3. MERGE & NORMALIZE
# ==========================================
if len(X_list) == 0:
    raise ValueError("No valid training data found! Check your file paths.")

X_data = np.vstack(X_list)
y_data = np.concatenate(y_list)

print(f"\n🔄 Total Pixels Loaded: {X_data.shape[0]:,}")

# Downsample for speed (Neural Nets are slower than Random Forest)
if len(y_data) > 300000:
    print("   Downsampling to 300,000 pixels for faster training...")
    idx = np.random.choice(len(y_data), 300000, replace=False)
    X_data = X_data[idx]
    y_data = y_data[idx]

# Neural Networks require data to be scaled (0-1 range roughly)
print("⚖️ Normalizing data (StandardScaler)...")
scaler = StandardScaler()
X_data = scaler.fit_transform(X_data)

# Split
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.2, random_state=42)

# ==========================================
# 4. TRAIN WITH EPOCHS (Neural Network)
# ==========================================
print(f"\n🧠 Training Neural Network (MLP) on {len(X_train):,} pixels...")
print("   (This will print progress for each Epoch/Iteration)")

# MLPClassifier is a Neural Network.
# max_iter = 30 means 30 Epochs.
# verbose = True will print the "Iteration" log.
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64), # Deep Learning layers
    activation='relu',
    solver='adam',
    max_iter=30,     # <--- EPOCHS
    batch_size=256,
    random_state=42,
    verbose=True     # <--- SHOW PROGRESS
)

mlp.fit(X_train, y_train)

# ==========================================
# 5. VALIDATE
# ==========================================
print("\n🔍 Evaluating Model...")
y_pred = mlp.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print("-" * 40)
print(f"✅ Final Test Accuracy: {acc * 100:.2f}%")
print("-" * 40)
print(classification_report(y_test, y_pred))

In [ ]:
import rasterio
from rasterio.windows import from_bounds
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

# ==========================================
# 1. FILE SETUP
# ==========================================
sentinel_files = [
    "Sentinel2_Karnataka_2021-0000000000-0000011776.tif",
    "Sentinel2_Karnataka_2021-0000011776-0000000000.tif",
    "Sentinel2_Karnataka_2021-0000011776-0000011776.tif",
    "Sentinel2_Karnataka_2021-0000000000-0000000000.tif"
]

label_files = ["WorldCover_Karnataka_2021.tif", "SouthIndia_WorldCover.tif"]

# ==========================================
# 2. FEATURE ENGINEERING FUNCTION
# ==========================================
def add_indices(X):
    """
    Calculates NDVI and NDWI to help the model distinguish
    Vegetation (Forest/Crops) and Water.
    Assumes bands are [Blue, Green, Red, NIR]
    """
    # Prevent division by zero
    epsilon = 1e-8

    blue  = X[:, 0]
    green = X[:, 1]
    red   = X[:, 2]
    nir   = X[:, 3]

    # 1. NDVI (Vegetation Health) = (NIR - Red) / (NIR + Red)
    ndvi = (nir - red) / (nir + red + epsilon)

    # 2. NDWI (Water Content) = (Green - NIR) / (Green + NIR)
    ndwi = (green - nir) / (green + nir + epsilon)

    # 3. Add these new 'bands' to the dataset
    # Stack columns: [Blue, Green, Red, NIR, NDVI, NDWI]
    return np.column_stack((X, ndvi, ndwi))

# ==========================================
# 3. DATA LOADING LOOP
# ==========================================
X_list = []
y_list = []

print("🚀 Starting Advanced Data Extraction...")

for s_file in sentinel_files:
    try:
        with rasterio.open(s_file) as src_img:
            # Read Sentinel Data
            img_data = src_img.read()
            bands, height, width = img_data.shape
            X_tile = img_data.reshape(bands, -1).T
            bounds = src_img.bounds

            # Try to match with any available label file
            found = False
            for l_file in label_files:
                try:
                    with rasterio.open(l_file) as src_label:
                        window = from_bounds(bounds.left, bounds.bottom, bounds.right, bounds.top, src_label.transform)
                        y_tile = src_label.read(1, window=window, out_shape=(height, width), boundless=True, fill_value=0)

                        if np.any(y_tile > 0):
                            y_tile_flat = y_tile.flatten()

                            # Filter: Labels > 0 AND Image pixels > 0
                            valid_mask = (y_tile_flat > 0) & (np.max(X_tile, axis=1) > 0)

                            if np.sum(valid_mask) > 0:
                                X_clean = X_tile[valid_mask]
                                y_clean = y_tile_flat[valid_mask]

                                # ADD NEW FEATURES (NDVI, NDWI)
                                X_enhanced = add_indices(X_clean)

                                X_list.append(X_enhanced)
                                y_list.append(y_clean)

                                print(f"   ✅ Processed {s_file} (Matched with {l_file})")
                                found = True
                                break
                except Exception:
                    continue

            if not found:
                print(f"   ⚠️ No matching labels found for {s_file}")

    except FileNotFoundError:
        pass

# ==========================================
# 4. PREPARE DATASET
# ==========================================
if not X_list:
    raise ValueError("No matching data found. Please check your files.")

X_data = np.vstack(X_list)
y_data = np.concatenate(y_list)

print(f"\n📊 Total Pixels: {X_data.shape[0]:,}")
print(f"   Features per pixel: {X_data.shape[1]} (Blue, Green, Red, NIR, NDVI, NDWI)")

# Downsample (Keep it manageable but large enough for accuracy)
if len(y_data) > 300000:
    print("   Downsampling to 300,000 balanced pixels...")
    # We use a simple random sample here, but stratify later
    idx = np.random.choice(len(y_data), 300000, replace=False)
    X_data = X_data[idx]
    y_data = y_data[idx]

print("⚖️ Scaling Data...")
scaler = StandardScaler()
X_data = scaler.fit_transform(X_data)

# Split (Stratify guarantees we keep rare classes in test set)
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42, stratify=y_data
)

# ==========================================
# 5. TRAIN DEEP NEURAL NETWORK
# ==========================================
print(f"\n🧠 Training Deep Neural Network (200 Epochs Max)...")

# Improvements:
# 1. Deeper Layers: (256, 128, 64) -> More capacity to learn complex patterns
# 2. Max Iter: 200 -> Gives it time to converge
# 3. Learning Rate: 'adaptive' -> Slows down as it gets closer to the answer
mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    solver='adam',
    max_iter=200,          # Increased from 30
    batch_size=512,        # Larger batch for stability
    learning_rate='adaptive',
    random_state=42,
    early_stopping=True,   # Stop if it stops improving (saves time)
    n_iter_no_change=10,
    verbose=True           # Show the Epochs
)

mlp.fit(X_train, y_train)

# ==========================================
# 6. RESULTS
# ==========================================
print("\n🔍 Evaluating...")
y_pred = mlp.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("="*40)
print(f"✅ FINAL ACCURACY: {acc * 100:.2f}%")
print("="*40)

# Class Names for easier reading
class_names = {
    10: "Tree Cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
    50: "Built-up", 60: "Bare / sparse veg", 70: "Snow and Ice",
    80: "Water", 90: "Wetland", 95: "Mangroves"
}
unique_labels = np.unique(y_test)
names = [class_names.get(i, str(i)) for i in unique_labels]

print(classification_report(y_test, y_pred, target_names=names, zero_division=0))

🚀 Starting Advanced Data Extraction...
   ⚠️ No matching labels found for Sentinel2_Karnataka_2021-0000000000-0000011776.tif
   ✅ Processed Sentinel2_Karnataka_2021-0000011776-0000000000.tif (Matched with WorldCover_Karnataka_2021.tif)
   ⚠️ No matching labels found for Sentinel2_Karnataka_2021-0000011776-0000011776.tif
   ✅ Processed Sentinel2_Karnataka_2021-0000000000-0000000000.tif (Matched with WorldCover_Karnataka_2021.tif)

📊 Total Pixels: 4,982,931
   Features per pixel: 6 (Blue, Green, Red, NIR, NDVI, NDWI)
   Downsampling to 300,000 balanced pixels...
⚖️ Scaling Data...

🧠 Training Deep Neural Network (200 Epochs Max)...
Iteration 1, loss = 0.83305840
Validation score: 0.748667
Iteration 2, loss = 0.75577770
Validation score: 0.749292
Iteration 3, loss = 0.74914632
Validation score: 0.749958
Iteration 4, loss = 0.74489686
Validation score: 0.751125
Iteration 5, loss = 0.74306186
Validation score: 0.751583
Iteration 6, loss = 0.74080549
Validation score: 0.751958
Iteration 7, l

In [ ]:
import os
import random
import numpy as np
import rasterio
from rasterio.windows import Window, from_bounds
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# ==========================================
# 1. CONFIGURATION
# ==========================================
PATCH_SIZE = 128      # Input size for the U-Net (128x128 pixels)
NUM_PATCHES = 1000    # How many training samples to create (Increase to 5000+ for best results)
BATCH_SIZE = 16
EPOCHS = 30
NUM_CLASSES = 12      # WorldCover has ~11 classes, we leave room for 0-index

# File Paths (Update these if needed)
sentinel_files = [
    "Sentinel2_Karnataka_2021-0000000000-0000011776.tif",
    "Sentinel2_Karnataka_2021-0000011776-0000000000.tif",
    "Sentinel2_Karnataka_2021-0000011776-0000011776.tif",
    "Sentinel2_Karnataka_2021-0000000000-0000000000.tif"
]
label_file = "WorldCover_Karnataka_2021.tif"

# Class Mapping: WorldCover (10, 20...) -> Neural Net (0, 1...)
# We map 10->0, 20->1, etc. to make it compatible with Sparse Categorical Crossentropy
class_map = {
    10: 0, 20: 1, 30: 2, 40: 3, 50: 4,
    60: 5, 70: 6, 80: 7, 90: 8, 95: 9, 100: 10, 0: 11 # 0 is usually background
}

# ==========================================
# 2. DATA GENERATOR (Random Patch Cropping)
# ==========================================
def get_random_patches(sentinel_files, label_file, num_patches, patch_size):
    X_patches = []
    y_patches = []

    print(f"✂️  Generating {num_patches} patches of size {patch_size}x{patch_size}...")

    count = 0
    while count < num_patches:
        # Pick a random sentinel file
        s_file = random.choice(sentinel_files)

        try:
            with rasterio.open(s_file) as src_img:
                # Pick a random x, y coordinate
                w = src_img.width
                h = src_img.height

                if w < patch_size or h < patch_size: continue

                col_off = random.randint(0, w - patch_size)
                row_off = random.randint(0, h - patch_size)

                # Create a Window
                window = Window(col_off, row_off, patch_size, patch_size)

                # Read Sentinel Data (Features)
                # Shape: (Bands, 128, 128) -> Transpose to (128, 128, Bands)
                img = src_img.read(window=window)
                img = np.moveaxis(img, 0, -1)

                # Check for empty/black images
                if np.max(img) == 0: continue

                # Read Label Data (Targets)
                # We need the geographic bounds of this window to find it in the Label Map
                win_bounds = src_img.window_bounds(window)

                with rasterio.open(label_file) as src_lbl:
                    # Calculate window in label file based on geographic bounds
                    lbl_window = from_bounds(win_bounds[0], win_bounds[1], win_bounds[2], win_bounds[3], src_lbl.transform)
                    lbl = src_lbl.read(1, window=lbl_window, out_shape=(patch_size, patch_size))

                # Normalize Image (Sentinel is uint16 0-10000, we want 0-1)
                img = img.astype('float32') / 10000.0
                np.clip(img, 0, 1, out=img) # Clip max to 1.0

                # Remap Labels to 0-11
                lbl_remapped = np.vectorize(lambda x: class_map.get(x, 11))(lbl)

                X_patches.append(img)
                y_patches.append(lbl_remapped)

                count += 1
                if count % 100 == 0: print(f"   Generated {count}/{num_patches}...")

        except Exception as e:
            # Skip errors (usually bounds issues)
            continue

    return np.array(X_patches), np.array(y_patches)

# RUN GENERATOR
X_data, y_data = get_random_patches(sentinel_files, label_file, NUM_PATCHES, PATCH_SIZE)
print(f"✅ Data Ready. Shape: {X_data.shape}")

# Split into Train/Val
X_train, X_val, y_train, y_val = train_test_split(X_data, y_data, test_size=0.2, random_state=42)

# ==========================================
# 3. BUILD U-NET MODEL
# ==========================================
def simple_unet(input_size=(128, 128, 4), num_classes=12):
    inputs = layers.Input(input_size)

    # Encoder (Downsampling)
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    c3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c3)
    p3 = layers.MaxPooling2D((2, 2))(c3)

    # Bridge (Bottleneck)
    b = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(p3)
    b = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(b)

    # Decoder (Upsampling)
    u1 = layers.UpSampling2D((2, 2))(b)
    u1 = layers.Concatenate()([u1, c3])
    c4 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u1)
    c4 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c4)

    u2 = layers.UpSampling2D((2, 2))(c4)
    u2 = layers.Concatenate()([u2, c2])
    c5 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u2)
    c5 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c5)

    u3 = layers.UpSampling2D((2, 2))(c5)
    u3 = layers.Concatenate()([u3, c1])
    c6 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(u3)
    c6 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(c6)

    # Output Layer
    outputs = layers.Conv2D(num_classes, (1, 1), activation='softmax')(c6)

    model = models.Model(inputs=[inputs], outputs=[outputs])
    return model

# Initialize
model = simple_unet(input_size=(PATCH_SIZE, PATCH_SIZE, 4), num_classes=NUM_CLASSES)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("\n🧠 Model Architecture Built (U-Net).")

# ==========================================
# 4. TRAIN
# ==========================================
print("🚀 Starting Training...")
history = model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, y_val)
)

# ==========================================
# 5. VISUALIZE RESULTS
# ==========================================
print("\n🔍 Visualizing Predictions...")
def plot_prediction(model, X, y, index=0):
    # Predict
    pred_mask = model.predict(X[index:index+1])
    pred_mask = np.argmax(pred_mask, axis=-1)[0]

    fig, ax = plt.subplots(1, 3, figsize=(15, 5))

    # Show Sentinel Image (RGB only)
    # Bands are B2, B3, B4, B8. RGB is indices 2, 1, 0.
    rgb = X[index][:,:,[2,1,0]]
    # Brighten it a bit for display
    rgb = np.clip(rgb * 3.5, 0, 1)

    ax[0].imshow(rgb)
    ax[0].set_title("Sentinel-2 Input")

    ax[1].imshow(y[index], cmap='tab20')
    ax[1].set_title("True Label (WorldCover)")

    ax[2].imshow(pred_mask, cmap='tab20')
    ax[2].set_title("U-Net Prediction")

    plt.show()

# Plot a few examples
plot_prediction(model, X_val, y_val, index=0)
plot_prediction(model, X_val, y_val, index=5)

✂️  Generating 1000 patches of size 128x128...


In [ ]:
import numpy as np
import rasterio
from rasterio.windows import Window, from_bounds
import random
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models

# ==========================================
# 1. DEFINE FILES (So Python doesn't forget)
# ==========================================
sentinel_files = [
    "Sentinel2_Karnataka_2021-0000000000-0000011776.tif",
    "Sentinel2_Karnataka_2021-0000011776-0000000000.tif",
    "Sentinel2_Karnataka_2021-0000011776-0000011776.tif",
    "Sentinel2_Karnataka_2021-0000000000-0000000000.tif"
]

label_file = "WorldCover_Karnataka_2021.tif"

# ==========================================
# 2. BULLETPROOF DATA GENERATOR
# ==========================================
# Map classes to 0-10. Anything else (like 0 or 255) becomes Class 11.
SAFE_CLASS_MAP = {
    10: 0, 20: 1, 30: 2, 40: 3, 50: 4,
    60: 5, 70: 6, 80: 7, 90: 8, 95: 9, 100: 10
}

def get_clean_patches(sentinel_files, label_file, num_patches=200, patch_size=128):
    X_patches = []
    y_patches = []

    print(f"🛡️ Generating {num_patches} CLEAN patches (Sanity Check)...")

    count = 0
    max_retries = num_patches * 5
    retries = 0

    while count < num_patches and retries < max_retries:
        s_file = random.choice(sentinel_files)
        retries += 1

        try:
            with rasterio.open(s_file) as src_img:
                w, h = src_img.width, src_img.height
                if w < patch_size or h < patch_size: continue

                # Random Window
                col = random.randint(0, w - patch_size)
                row = random.randint(0, h - patch_size)
                window = Window(col, row, patch_size, patch_size)

                # 1. READ IMAGE
                img = src_img.read(window=window)
                img = np.moveaxis(img, 0, -1)

                # 2. FIX IMAGE MATH (Crucial Step)
                img = np.nan_to_num(img, nan=0.0, posinf=1.0, neginf=0.0)

                # Normalize (Sentinel 0-10000 -> 0-1)
                img = img.astype('float32') / 10000.0
                img = np.clip(img, 0, 1)

                if np.mean(img) == 0: continue

                # 3. READ LABELS
                win_bounds = src_img.window_bounds(window)
                with rasterio.open(label_file) as src_lbl:
                    lbl_window = from_bounds(win_bounds[0], win_bounds[1], win_bounds[2], win_bounds[3], src_lbl.transform)
                    lbl = src_lbl.read(1, window=lbl_window, out_shape=(patch_size, patch_size), boundless=True, fill_value=0)

                # 4. FIX LABEL MAPPING
                lbl_clean = np.vectorize(lambda x: SAFE_CLASS_MAP.get(x, 11))(lbl)

                X_patches.append(img)
                y_patches.append(lbl_clean)
                count += 1

        except Exception:
            continue

    return np.array(X_patches), np.array(y_patches)

# ==========================================
# 3. BUILD SIMPLE U-NET
# ==========================================
def simple_unet(input_size=(128, 128, 4), num_classes=12):
    inputs = layers.Input(input_size)

    # Simple Encoder
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(p1)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    # Bridge
    b = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p2)

    # Simple Decoder
    u1 = layers.UpSampling2D((2, 2))(b)
    c3 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u1)

    u2 = layers.UpSampling2D((2, 2))(c3)
    c4 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(u2)

    outputs = layers.Conv2D(num_classes, (1, 1), activation='softmax')(c4)
    return models.Model(inputs=[inputs], outputs=[outputs])

# ==========================================
# 4. RUN THE SANITY CHECK
# ==========================================
try:
    # Generate small data
    X_safe, y_safe = get_clean_patches(sentinel_files, label_file, num_patches=200, patch_size=128)
    print(f"✅ Data Shape: {X_safe.shape}")

    # Split
    X_train_safe, X_val_safe, y_train_safe, y_val_safe = train_test_split(X_safe, y_safe, test_size=0.2)

    # Compile with LOW Learning Rate (Safety First)
    model = simple_unet(input_size=(128, 128, 4), num_classes=12)
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    print("\n🚀 Starting 2-Epoch Test Run...")
    model.fit(X_train_safe, y_train_safe, epochs=2, batch_size=16, validation_data=(X_val_safe, y_val_safe))
    print("\n✅ Check finished! If 'loss' is a number (not nan), you are safe to run the big training.")

except Exception as e:
    print(f"\n❌ Something went wrong: {e}")

In [ ]:
import rasterio
import os
from rasterio.windows import from_bounds
from rasterio.vrt import WarpedVRT

# 1. SETUP FILES
sentinel_files = [
    "Sentinel2_Karnataka_2021-0000000000-0000011776.tif",
    "Sentinel2_Karnataka_2021-0000011776-0000000000.tif",
    "Sentinel2_Karnataka_2021-0000011776-0000011776.tif",
    "Sentinel2_Karnataka_2021-0000000000-0000000000.tif"
]
label_file = "WorldCover_Karnataka_2021.tif"

print("🔍 DIAGNOSTIC MODE: Checking Coordinate Systems (CRS)...")

# 2. CHECK ONE SENTINEL FILE
s_file = sentinel_files[1] # Pick one that likely exists
if not os.path.exists(s_file):
    print(f"❌ File not found: {s_file}")
else:
    with rasterio.open(s_file) as src_s:
        print(f"\n🛰️ SATELLITE IMAGE:")
        print(f"   Name: {s_file}")
        print(f"   CRS (System): {src_s.crs}")
        print(f"   Bounds: {src_s.bounds}")

        # 3. CHECK LABEL FILE
        if not os.path.exists(label_file):
             print(f"❌ Label file not found: {label_file}")
        else:
            with rasterio.open(label_file) as src_l:
                print(f"\n🗺️ LABEL MAP:")
                print(f"   CRS (System): {src_l.crs}")
                print(f"   Bounds: {src_l.bounds}")

                # 4. THE CRITICAL CHECK
                print(f"\n⚖️ COMPARISON:")
                if src_s.crs == src_l.crs:
                    print("   ✅ CRS Matches! (This is good)")

                    # Check if they physically overlap
                    s_left, s_bottom, s_right, s_top = src_s.bounds
                    l_left, l_bottom, l_right, l_top = src_l.bounds

                    if (s_left > l_right or s_right < l_left or s_top < l_bottom or s_bottom > l_top):
                        print("   ❌ ERROR: NO SPATIAL OVERLAP.")
                        print("   The images are in the same system, but show different parts of the world.")
                    else:
                        print("   ✅ SPATIAL OVERLAP CONFIRMED.")
                else:
                    print("   ❌ CRS MISMATCH DETECTED!")
                    print(f"   Satellite is {src_s.crs} (likely Meters)")
                    print(f"   Label is {src_l.crs} (likely Degrees/LatLon)")
                    print("   ⚠️ This is why you got 0 patches. The code is looking for Lat/Lon coordinates inside a Meter grid.")

In [ ]:
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling

with rasterio.open(s_file) as src_s:
    sentinel_crs = src_s.crs
    sentinel_transform = src_s.transform
    sentinel_width = src_s.width
    sentinel_height = src_s.height

    with rasterio.open(label_file) as src_l:
        with WarpedVRT(
            src_l,
            crs=sentinel_crs,
            transform=sentinel_transform,
            width=sentinel_width,
            height=sentinel_height,
            resampling=Resampling.nearest  # IMPORTANT for labels
        ) as label_vrt:

            # Test read
            label_data = label_vrt.read(1)
            print("✅ Label aligned to Sentinel grid")
            print("Unique classes:", set(label_data.flatten()))


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models

import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt


In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [ ]:
import numpy as np

X_patches = np.array(X_patches, dtype=np.float32)
y_patches = np.array(y_patches, dtype=np.uint8)

print(X_patches.shape, y_patches.shape)



In [ ]:
import rasterio
from rasterio.windows import Window
import os
import numpy as np

IMG_PATH = "karnataka_s2_20m.tif"
MASK_PATH = "karnataka_lulc_mask_20m.tif"

OUT_IMG_DIR = "tiles/images"
OUT_MASK_DIR = "tiles/masks"

os.makedirs(OUT_IMG_DIR, exist_ok=True)
os.makedirs(OUT_MASK_DIR, exist_ok=True)

TILE_SIZE = 256
stride = 256

with rasterio.open(IMG_PATH) as img_ds, rasterio.open(MASK_PATH) as mask_ds:
    width = img_ds.width
    height = img_ds.height

    tile_id = 0
    for y in range(0, height - TILE_SIZE, stride):
        for x in range(0, width - TILE_SIZE, stride):
            window = Window(x, y, TILE_SIZE, TILE_SIZE)

            img_tile = img_ds.read(window=window)
            mask_tile = mask_ds.read(1, window=window)

            # skip empty tiles
            if np.all(mask_tile == 0):
                continue

            img_meta = img_ds.meta.copy()
            img_meta.update({
                "height": TILE_SIZE,
                "width": TILE_SIZE,
                "transform": rasterio.windows.transform(window, img_ds.transform)
            })

            mask_meta = mask_ds.meta.copy()
            mask_meta.update({
                "height": TILE_SIZE,
                "width": TILE_SIZE,
                "transform": rasterio.windows.transform(window, mask_ds.transform),
                "count": 1
            })

            with rasterio.open(f"{OUT_IMG_DIR}/tile_{tile_id}.tif", "w", **img_meta) as f:
                f.write(img_tile)

            with rasterio.open(f"{OUT_MASK_DIR}/tile_{tile_id}.tif", "w", **mask_meta) as f:
                f.write(mask_tile, 1)

            tile_id += 1

print("✅ Tiling completed:", tile_id, "tiles created")


RasterioIOError: karnataka_s2_20m.tif: No such file or directory